In [ ]:
!pip install -q "transformers>=4.44" "datasets>=2.20" bitsandbytes evaluate rouge_score gptqmodel optimum

# 1. DATA PIPELINE

In [ ]:
from datasets import load_dataset

raw = load_dataset("lighteval/med_paragraph_simplification")

print("splits + sizes:", {k: len(v) for k, v in raw.items()})
first = next(iter(raw.values()))
print("columns:", first.column_names)
print("\none example:")
print(first[0])

splits + sizes: {'train': 3568, 'validation': 411, 'test': 480}
columns: ['query', 'answer']

one example:
{'query': 'Two trials met the inclusion criteria. One compared 2% ketanserin ointment in polyethylene glycol (PEG) with PEG alone, used twice a day by 40 participants with arterial leg ulcers, for eight weeks or until healing, whichever was sooner. One compared topical application of blood-derived concentrated growth factor (CGF) with standard dressing (polyurethane film or foam); both applied weekly for six weeks by 61 participants with non-healing ulcers (venous, diabetic arterial, neuropathic, traumatic, or vasculitic). Both trials were small, reported results inadequately, and were of low methodological quality. Short follow-up times (six and eight weeks) meant it would be difficult to capture sufficient healing events to allow us to make comparisons between treatments. One trial demonstrated accelerated wound healing in the ketanserin group compared with the control group. In

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

print("vocab size:", tok.vocab_size, "| pad:", tok.pad_token, "| eos:", tok.eos_token)

vocab size: 151643 | pad: <|endoftext|> | eos: <|im_end|>


In [ ]:
PROMPT_TEMPLATE = (
    "Simplify the following medical text into plain language a patient can understand.\n\n"
    "Medical text:\n{technical}\n\nPlain language:\n"
)

ex = raw["train"][0]
prompt = PROMPT_TEMPLATE.format(technical=ex["query"])
full_text = prompt + ex["answer"] + tok.eos_token   # what the model will learn from: instruction+input, then the answer

enc = tok(full_text)

print("num tokens:", len(enc["input_ids"]))
print("first 20 ids:", enc["input_ids"][:20])
print("round-trip decode:", tok.decode(enc["input_ids"][:20]))

num tokens: 878
first 20 ids: [50, 70206, 279, 2701, 6457, 1467, 1119, 14396, 4128, 264, 8720, 646, 3535, 382, 52554, 1467, 510, 11613, 19080, 2270]
round-trip decode: Simplify the following medical text into plain language a patient can understand.

Medical text:
Two trials met


In [ ]:
import torch
from torch.utils.data import Dataset

MAX_LEN = 512

class MedSimplifyDataset(Dataset):
    def __init__(self, split):
        self.data = split          # a HF split, e.g. raw["train"]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # TODO(you): same idea as Cell 4, but:
        #   1. ex = self.data[idx]; build prompt + full_text (as in Cell 4)
        #   2. tokenize WITH truncation=True, max_length=MAX_LEN   <- so the 878-token ones fit
        #   3. return a dict of 1-D tensors:
        #        {"input_ids": <tensor>, "attention_mask": <tensor>}
        #      The tokenizer gives python LISTS; wrap each with torch.tensor(..., dtype=torch.long)
        ex = self.data[idx]
        prompt = PROMPT_TEMPLATE.format(technical=ex["query"])
        full_text = prompt + ex["answer"] + tok.eos_token
        enc = tok(full_text, truncation=True, max_length=MAX_LEN)
        return {
            "input_ids" : torch.tensor(enc["input_ids"]),
            "attention_mask" : torch.tensor(enc["attention_mask"])
        }

# quick self-test:
ds = MedSimplifyDataset(raw["train"])
print("dataset size:", len(ds))
item = ds[0]
print("keys:", item.keys())
print("input_ids:", item["input_ids"].shape, item["input_ids"].dtype)

dataset size: 3568
keys: dict_keys(['input_ids', 'attention_mask'])
input_ids: torch.Size([512]) torch.int64


In [ ]:
from torch.utils.data import DataLoader

ds = MedSimplifyDataset(raw["train"])
bad_loader = DataLoader(ds, batch_size=4)   # no collate_fn yet

try:
    batch = next(iter(bad_loader))
    print("input_ids:", batch["input_ids"].shape)
except Exception as e:
    print("ERROR:", type(e).__name__, "-", str(e)[:160])

ERROR: RuntimeError - stack expects each tensor to be equal size, but got [512] at entry 0 and [456] at entry 1


In [ ]:
from dataclasses import dataclass

@dataclass
class PadCollator:
    pad_token_id: int

    def __call__(self, batch):
        # batch = list of dicts, each {"input_ids":[L_i], "attention_mask":[L_i]} with DIFFERENT L_i
        max_len = max(example["input_ids"].size(0) for example in batch)

        input_ids, masks = [], []
        for ex in batch:
            ids, mask = ex["input_ids"], ex["attention_mask"]
            pad_len = max_len - ids.size(0)          # how many pad slots this example needs

            # TODO: right-pad `ids` with self.pad_token_id, and `mask` with 0, each by pad_len.
            #   hint: torch.cat([ids,  torch.full((pad_len,), self.pad_token_id, dtype=torch.long)])
            #         torch.cat([mask, torch.zeros(pad_len, dtype=torch.long)])
            ids  = torch.cat([ids, torch.full((pad_len, ), self.pad_token_id, dtype=torch.long)])
            mask = torch.cat([mask, torch.zeros(pad_len, dtype=torch.long)])

            input_ids.append(ids)
            masks.append(mask)

        # TODO: stack the lists of 1-D tensors into 2-D [B, max_len] tensors (hint: torch.stack)
        return {"input_ids": torch.stack(input_ids), "attention_mask": torch.stack(masks)}


# test it with a real DataLoader:
collator = PadCollator(pad_token_id=tok.pad_token_id)
loader = DataLoader(ds, batch_size=4, shuffle=True, collate_fn=collator)
batch = next(iter(loader))
print("input_ids:", batch["input_ids"].shape)       # expect [4, max_len_in_batch]
print("attention_mask:", batch["attention_mask"].shape)
print("mask row 0 (1=real, 0=pad):", batch["attention_mask"][0].tolist()[-10:])

input_ids: torch.Size([4, 512])
attention_mask: torch.Size([4, 512])
mask row 0 (1=real, 0=pad): [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [ ]:
def make_loaders(batch_size=4):
    collator = PadCollator(pad_token_id=tok.pad_token_id)
    train_dl = DataLoader(MedSimplifyDataset(raw["train"]),
                          batch_size=batch_size, shuffle=True,  collate_fn=collator)
    val_dl   = DataLoader(MedSimplifyDataset(raw["validation"]),
                          batch_size=batch_size, shuffle=False, collate_fn=collator)
    return train_dl, val_dl

train_dl, val_dl = make_loaders()
print("train batches:", len(train_dl), "| val batches:", len(val_dl))
xb = next(iter(train_dl))
print("one batch:", xb["input_ids"].shape)

train batches: 892 | val batches: 103
one batch: torch.Size([4, 512])


# 2. FINE TUNING

In [ ]:
# Load the model
from transformers import AutoModelForCausalLM
import torch

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,   # half precision -> fits the T4, trains faster
    device_map="cuda",           # put it on the GPU
)
n_params = sum(p.numel() for p in model.parameters())
print(f"{model.config.model_type} loaded | {n_params/1e9:.2f}B params | device: {next(model.parameters()).device}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

qwen2 loaded | 1.54B params | device: cuda:0


In [ ]:
batch = next(iter(train_dl))
batch = {k: v.to("cuda") for k, v in batch.items()}   # move the batch onto the GPU (model is there)

out = model(
    input_ids=batch["input_ids"],
    attention_mask=batch["attention_mask"],
    labels=batch["input_ids"]
)

print("loss:", out.loss.item())
print("logits shape:", out.logits.shape)   # [batch, seq_len, vocab_size]

loss: 2.1208314895629883
logits shape: torch.Size([4, 512, 151936])


In [ ]:
!pip uninstall -y torchao

Found existing installation: torchao 0.18.0
Uninstalling torchao-0.18.0:
  Successfully uninstalled torchao-0.18.0


In [ ]:
!pip install -q peft

In [ ]:
from peft import LoraConfig, get_peft_model

lora_cfg = LoraConfig(
    r=8,                  # adapter "rank" — bigger = more capacity + more params. 8 is a fine default.
    lora_alpha=16,        # a scaling factor on the adapters
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # inject adapters into the attention layers
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()


INFO  LogBar: headless/CI mode; progress animations disabled.                   
WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.
INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7' for memory saving.
INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.           
INFO  

┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐
│ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │
└─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘
GPT-QModel   : 7.6.0
Transformers : 5.18.0
Torch        : 2.11.0+cu130
Triton       : 3.6.0
trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410


In [ ]:
from torch.amp import autocast, GradScaler

# memory savers (you OOM'd before — these prevent it):
model.gradient_checkpointing_enable()   # recompute instead of storing activations
model.enable_input_require_grads()
model.config.use_cache = False          # required alongside gradient checkpointing

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=2e-4)
scaler = GradScaler("cuda")             # scales fp16 gradients so they don't vanish to 0
model.train()                           # put model in training mode
print("ready to train")

ready to train


In [ ]:
def make_labels(input_ids, attention_mask):
    labels = input_ids.clone()
    labels[attention_mask == 0] = -100     # -100 tells the loss to IGNORE padding positions
    return labels

MAX_STEPS = 60
step = 0
for batch in train_dl:
    batch = {k: v.to("cuda") for k, v in batch.items()}
    labels = make_labels(batch["input_ids"], batch["attention_mask"])

    optimizer.zero_grad()
    with autocast("cuda", dtype=torch.float16):
      out = model(
          input_ids=batch["input_ids"],
          attention_mask=batch["attention_mask"],
          labels=labels
      )
      loss = out.loss

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    step += 1
    if step % 10 == 0:
        print(f"step {step:3d} | loss {loss.item():.3f}")
    if step >= MAX_STEPS:
        break
print("training done")

step  10 | loss 1.630
step  20 | loss 1.496
step  30 | loss 1.607
step  40 | loss 1.718
step  50 | loss 1.745
step  60 | loss 1.631
training done


In [ ]:
model.eval()                              # eval mode (turns off dropout etc.)
model.gradient_checkpointing_disable()
model.config.use_cache = True             # re-enable the KV cache for fast generation — Day 3 is ALL about this

val_ex = raw["validation"][0]
prompt = PROMPT_TEMPLATE.format(technical=val_ex["query"])   # note: prompt only, NO answer this time
inputs = tok(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():                     # no training here -> don't track gradients (faster, less memory)
    gen = model.generate(**inputs, max_new_tokens=200, do_sample=False)

new_tokens = gen[0][inputs["input_ids"].shape[1]:]   # slice off the prompt; keep only what the model generated
print("=== MODEL OUTPUT ===")
print(tok.decode(new_tokens, skip_special_tokens=True))
print("\n=== HUMAN REFERENCE ===")
print(val_ex["answer"][:500])

=== MODEL OUTPUT ===
Interferon is an antiviral drug used to treat hepatitis C virus infection. It has been shown to be effective in reducing the amount of virus in the blood and improving liver function tests. This review found that interferon is also effective at curing hepatitis C virus infection. In addition, it improves liver function tests and reduces the risk of developing complications such as cirrhosis. There is some evidence that interferon may reduce the risk of developing liver cancer. Interferon is most effective when given over a long period of time and at high doses. Side effects include flu-like symptoms, fatigue, nausea, vomiting, headache, muscle pain, joint pain, fever, chills, and itching. These side effects tend to occur more frequently with longer courses of treatment. Interferon is not recommended for use in people who have already developed cirrhosis because there is a small chance that it could cause further damage to their liver. Interferon should only be cons

In [ ]:
import os
model.save_pretrained("lucidrt-lora")
tok.save_pretrained("lucidrt-lora")
print("saved:", os.listdir("lucidrt-lora"))
size_mb = sum(os.path.getsize(os.path.join("lucidrt-lora", f)) for f in os.listdir("lucidrt-lora")) / 1e6
print(f"adapter size: {size_mb:.1f} MB")   # tiny! just the adapters, not the 1.5B base

saved: ['tokenizer.json', 'README.md', 'adapter_model.safetensors', 'adapter_config.json', 'tokenizer_config.json', 'chat_template.jinja']
adapter size: 20.2 MB


# 3. KV Caching

In [ ]:
import time

# naive generation
def generate_naive(prompt, max_new_tokens=50):
    model.eval()
    input_ids = tok(prompt, return_tensors="pt").input_ids.to("cuda")

    for _ in range(max_new_tokens):
        with torch.no_grad():
            out = model(input_ids=input_ids, use_cache=False)   # recompute the WHOLE sequence every step
        logits = out.logits
        next_logits = logits[:, -1, :]
        next_id = next_logits.argmax(dim=-1, keepdim=True)
        input_ids = torch.cat([input_ids, next_id], dim=1)

    return input_ids

prompt = PROMPT_TEMPLATE.format(technical=raw["validation"][0]["query"])

t0 = time.time()
out_ids = generate_naive(prompt, max_new_tokens=50)
dt = time.time() - t0
print(f"50 tokens in {dt:.2f}s  ->  {50/dt:.1f} tokens/sec")
print("---")
print(tok.decode(out_ids[0], skip_special_tokens=True)[-300:])

50 tokens in 6.58s  ->  7.6 tokens/sec
---
anguage:
Interferon is effective in achieving viral clearance and improving liver biochemistry and histology in interferon naive patients with chronic hepatitis C. Higher doses and prolonged durations are more effective, but associated with more frequent adverse events. Interferon is associated with


In [ ]:
def generate_cached(prompt, max_new_tokens=50):
    model.eval()
    input_ids = tok(prompt, return_tensors="pt").input_ids.to("cuda")

    generated = input_ids
    cur = input_ids        # first pass: feed the whole prompt
    past = None            # the KV cache; empty to start

    for _ in range(max_new_tokens):
        with torch.no_grad():
            out = model(input_ids=cur, past_key_values=past, use_cache=True)

        past = out.past_key_values
        next_id = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
        generated = torch.cat([generated, next_id], dim=1)
        cur = next_id  # next pass feeds only the new token, the cache supplies all the earlier ones

    return generated

In [ ]:
# benchmark: naive vs cached, same prompt, same output length
prompt = PROMPT_TEMPLATE.format(technical=raw["validation"][0]["query"])

t0 = time.time(); naive_ids  = generate_naive(prompt, 50);  t_naive  = time.time() - t0
t0 = time.time(); cached_ids = generate_cached(prompt, 50); t_cached = time.time() - t0

print(f"naive :  {t_naive:.2f}s  ->  {50/t_naive:.1f} tok/s")
print(f"cached:  {t_cached:.2f}s  ->  {50/t_cached:.1f} tok/s")
print(f"speedup: {t_naive/t_cached:.2f}x")
print("same output? ", torch.equal(naive_ids, cached_ids))   # should be True — same tokens, just faster

naive :  11.43s  ->  4.4 tok/s
cached:  9.33s  ->  5.4 tok/s
speedup: 1.22x
same output?  True


# 4. Quantization

In [ ]:
from transformers import BitsAndBytesConfig

model_int8 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=BitsAndBytesConfig(load_in_8bit=True),
    # note: NO .to("cuda") — bitsandbytes places it on the GPU for you
)
model_int8.eval()

print(f"fp16 footprint: {model.get_memory_footprint()/1e9:.2f} GB")
print(f"int8 footprint: {model_int8.get_memory_footprint()/1e9:.2f} GB")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

fp16 footprint: 3.10 GB
int8 footprint: 1.78 GB


In [ ]:
def generate_cached(prompt, max_new_tokens=50, gen_model=None):
  gen_model = gen_model or model          # <- default to the fp16 model
  model.eval()
  input_ids = tok(prompt, return_tensors="pt").input_ids.to("cuda")

  generated = input_ids
  cur = input_ids        # first pass: feed the whole prompt
  past = None            # the KV cache; empty to start

  for _ in range(max_new_tokens):
      with torch.no_grad():
          out = gen_model(input_ids=cur, past_key_values=past, use_cache=True)

      past = out.past_key_values
      next_id = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
      generated = torch.cat([generated, next_id], dim=1)
      cur = next_id  # next pass feeds only the new token, the cache supplies all the earlier ones

  return generated

In [ ]:
  prompt = PROMPT_TEMPLATE.format(technical=raw["validation"][0]["query"])

  t0 = time.time(); fp16_ids = generate_cached(prompt, 50);                   t_fp16 = time.time() - t0
  t0 = time.time(); int8_ids = generate_cached(prompt, 50, gen_model=model_int8); t_int8 = time.time() - t0

  print(f"fp16 cached:  {t_fp16:.2f}s  ->  {50/t_fp16:.1f} tok/s")
  print(f"int8 cached:  {t_int8:.2f}s  ->  {50/t_int8:.1f} tok/s")
  print("outputs identical? ", torch.equal(fp16_ids, int8_ids))
  print("--- int8 output ---")
  print(tok.decode(int8_ids[0], skip_special_tokens=True)[-200:])

Streaming output truncated to the last 5000 lines.


fp16 cached:  3.17s  ->  15.8 tok/s
int8 cached:  33.11s  ->  1.5 tok/s
outputs identical?  False
--- int8 output ---
four studies involving 6,545 patients were included. Compared to no treatment, interferon, which is a medication, increased the chance of a positive result in a test for the virus (Peto odds ratio 4.6


In [ ]:
import time, torch

def profile_split(prompt, max_new_tokens=50):
    model.eval()
    input_ids = tok(prompt, return_tensors="pt").input_ids.to("cuda")
    n_prompt = input_ids.shape[1]

    # warm-up: first CUDA launch is slow; run once and throw it away
    with torch.no_grad():
        _ = model(input_ids=input_ids, use_cache=True)
    torch.cuda.synchronize()

    # ---- PREFILL: one pass over the WHOLE prompt ----
    torch.cuda.synchronize(); t0 = time.time()
    with torch.no_grad():
        out = model(input_ids=input_ids, use_cache=True)
    torch.cuda.synchronize(); t_prefill = time.time() - t0

    past = out.past_key_values
    cur  = out.logits[:, -1, :].argmax(-1, keepdim=True)

    # ---- DECODE: one token at a time, from the cache ----
    torch.cuda.synchronize(); t0 = time.time()
    for _ in range(max_new_tokens):
        with torch.no_grad():
            out = model(input_ids=cur, past_key_values=past, use_cache=True)
        past = out.past_key_values
        cur  = out.logits[:, -1, :].argmax(-1, keepdim=True)
    torch.cuda.synchronize(); t_decode = time.time() - t0

    total = t_prefill + t_decode
    print(f"prompt length:        {n_prompt} tokens")
    print(f"prefill:            {t_prefill*1000:8.1f} ms  ({t_prefill/total*100:4.1f}%)  for all {n_prompt} tokens")
    print(f"decode ({max_new_tokens} tokens):   {t_decode*1000:8.1f} ms  ({t_decode/total*100:4.1f}%)")
    print(f"  per prompt token (prefill): {t_prefill/n_prompt*1000:6.3f} ms")
    print(f"  per output token (decode):  {t_decode/max_new_tokens*1000:6.3f} ms")

prompt = PROMPT_TEMPLATE.format(technical=raw["validation"][0]["query"])
profile_split(prompt, max_new_tokens=50)

prompt length:        448 tokens
prefill:               126.9 ms  ( 2.8%)  for all 448 tokens
decode (50 tokens):     4392.7 ms  (97.2%)
  per prompt token (prefill):  0.283 ms
  per output token (decode):  87.854 ms


In [ ]:
profile_split(prompt, max_new_tokens=200)

prompt length:        448 tokens
prefill:               116.5 ms  ( 0.5%)  for all 448 tokens
decode (200 tokens):    22105.8 ms  (99.5%)
  per prompt token (prefill):  0.260 ms
  per output token (decode):  110.529 ms


In [ ]:
def batched_throughput(prompt, batch_size, max_new_tokens=50):
    model.eval()
    ids = tok(prompt, return_tensors="pt").input_ids.to("cuda")
    input_ids = ids.repeat(batch_size, 1)        # [B, L] — B identical prompts

    try:
        torch.cuda.synchronize(); t0 = time.time()
        cur, past = input_ids, None
        for _ in range(max_new_tokens):
            with torch.no_grad():
                out = model(input_ids=cur, past_key_values=past, use_cache=True)
            past = out.past_key_values
            cur  = out.logits[:, -1, :].argmax(-1, keepdim=True)   # [B, 1]
        torch.cuda.synchronize(); dt = time.time() - t0

        total = batch_size * max_new_tokens
        print(f"batch={batch_size:2d} | {dt:5.2f}s | {total/dt:7.1f} tok/s TOTAL | {max_new_tokens/dt:5.1f} tok/s per-seq")
    except torch.cuda.OutOfMemoryError:
        print(f"batch={batch_size:2d} | OOM — hit the memory ceiling")
        torch.cuda.empty_cache()

prompt = PROMPT_TEMPLATE.format(technical=raw["validation"][0]["query"])
for b in [1, 2, 4, 8, 16]:
    batched_throughput(prompt, b, max_new_tokens=50)

batch= 1 |  4.47s |    11.2 tok/s TOTAL |  11.2 tok/s per-seq
batch= 2 |  4.60s |    21.7 tok/s TOTAL |  10.9 tok/s per-seq
batch= 4 |  4.66s |    42.9 tok/s TOTAL |  10.7 tok/s per-seq
batch= 8 |  5.23s |    76.4 tok/s TOTAL |   9.6 tok/s per-seq
batch=16 |  6.46s |   123.8 tok/s TOTAL |   7.7 tok/s per-seq


In [ ]:
import evaluate
rouge = evaluate.load("rouge")

In [ ]:
def eval_quality(gen_model, n=40, max_new_tokens=64, batch_size=8):
    tok.padding_side = "left"   # decoder-only models MUST left-pad for batched generation,
                                # so every sequence's newest token sits at the right edge
    data = raw["validation"].select(range(n))
    preds, refs = [], []

    for i in range(0, n, batch_size):
        chunk   = data.select(range(i, min(i + batch_size, n)))
        prompts = [PROMPT_TEMPLATE.format(technical=ex["query"]) for ex in chunk]
        enc = tok(prompts, return_tensors="pt", padding=True,
                  truncation=True, max_length=512).to("cuda")

        with torch.no_grad():
            out = gen_model.generate(**enc, max_new_tokens=max_new_tokens,
                                      do_sample=False,              # greedy = matches our argmax loop
                                      pad_token_id=tok.pad_token_id)

        gen_only = out[:, enc["input_ids"].shape[1]:]              # strip the prompt, keep new tokens
        preds.extend(tok.batch_decode(gen_only, skip_special_tokens=True))
        refs.extend([ex["answer"] for ex in chunk])

    scores = rouge.compute(predictions=preds, references=refs)
    return {k: round(v, 4) for k, v in scores.items()}

# print("fp16:", eval_quality(model,      n=40))
# print("int8:", eval_quality(model_int8, n=40))

In [ ]:
!pip install -q autoawq

In [ ]:
model_awq = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct-AWQ",
    device_map="cuda",
)

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.61GB            

model.safetensors: downloading bytes:           |  0.00B            

INFO  ExLlamaV2 AWQ: compiling torch.ops JIT extension in `/root/.cache/gptqmodel/torch_extensions/exllamav2_awq/28102340d90d94d7`.
INFO  ExLlamaV2 AWQ: torch.ops JIT extension ready in 42s (estimated ~35s, +6.7s).
INFO  Kernel: Auto-selection: adding candidate `AwqExllamaV2Linear`             
INFO  Kernel: selected -> `AwqExllamaV2Linear`.                                 


Loading weights:   0%|          | 0/731 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

INFO  gc.collect() reclaimed 12584 objects in 0.364s                            


In [ ]:
prompt = PROMPT_TEMPLATE.format(technical=raw["validation"][0]["query"])

for name, m in [("fp16", model), ("int8", model_int8), ("int4-awq", model_awq)]:
  torch.cuda.synchronize(); t0 = time.time()
  _ = generate_cached(prompt, 50, gen_model=m)
  torch.cuda.synchronize(); dt = time.time() - t0
  print(f"{name:9s}: {dt:5.2f}s  ->  {50/dt:5.1f} tok/s")

fp16     :  3.11s  ->   16.1 tok/s


Streaming output truncated to the last 5000 lines.


int8     : 22.72s  ->    2.2 tok/s
int4-awq :  4.03s  ->   12.4 tok/s


In [ ]:
print("int4-awq:", eval_quality(model_awq, n=40))

int4-awq: {'rouge1': np.float64(0.1507), 'rouge2': np.float64(0.029), 'rougeL': np.float64(0.0969), 'rougeLsum': np.float64(0.1007)}


In [ ]:
print(f"fp16 footprint: {model.get_memory_footprint()/1e9:.2f} GB")
print(f"int4 footprint: {model_awq.get_memory_footprint()/1e9:.2f} GB")

fp16 footprint: 3.10 GB
int4 footprint: 1.15 GB


In [ ]:
prompt = PROMPT_TEMPLATE.format(technical=raw["validation"][0]["query"])
for name, m in [("fp16", model), ("int8", model_int8), ("int4-awq", model_awq)]:
  torch.cuda.synchronize(); t0 = time.time()
  _ = generate_cached(prompt, 50, gen_model=m)
  torch.cuda.synchronize(); dt = time.time() - t0
  print(f"{name:9s}: {dt:5.2f}s  ->  {50/dt:5.1f} tok/s")

fp16     :  8.76s  ->    5.7 tok/s


Streaming output truncated to the last 5000 lines.


int8     : 29.32s  ->    1.7 tok/s
int4-awq :  3.82s  ->   13.1 tok/s
